# IR System — Additional Features
**Step 10:** Query Refinement · Documents Clustering · RAG · LTR

⚠️ Use **GPU runtime** (Runtime → Change runtime type → T4 GPU)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/ir_system_data'
import os, sys

if not os.path.exists('/content/ir-system'):
    !git clone https://github.com/ghazal-mohammad/ir-system.git /content/ir-system
else:
    !cd /content/ir-system && git pull
sys.path.insert(0, '/content/ir-system')

!pip install ir-datasets==0.5.9 sentence-transformers==2.7.0 scikit-learn -q
!pip install transformers accelerate -q
print('ready')

## Part 1 — Query Refinement

In [ ]:
import json
import ir_datasets
from services.indexing_service import load_index
from services.preprocessing_service import preprocess
from services.query_refinement_service import (
    build_vocab_from_index, spell_correct_query,
    expand_query, suggest_related_terms, refine_query
)

index1 = load_index(f'{SAVE_DIR}/ct2021_index.pkl')
vocab1 = build_vocab_from_index(index1)
print(f'Vocabulary size: {len(vocab1):,}')

In [ ]:
# Test query refinement
test_queries = [
    'diabtes treatment for elderley patients',
    'heart failure medicaton dosage',
    'cancer immunotherapy clinical trial'
]

for raw_q in test_queries:
    tokens = preprocess(raw_q)
    result = refine_query(tokens, vocab1, index1, use_spell=True, use_expand=True)
    print(f'Original:   {raw_q}')
    print(f'Tokens:     {tokens}')
    print(f'Corrections:{result["corrections"]}')
    print(f'Synonyms:   {result["added_synonyms"]}')
    print(f'Expanded:   {result["expanded_tokens"]}')
    print()

# Related terms suggestion
tokens_test = preprocess('diabetes treatment')
related = suggest_related_terms(tokens_test, index1, {}, top_k=8)
print(f'Related terms for "diabetes treatment": {related}')

## Part 2 — Documents Clustering

In [ ]:
import numpy as np
from services.embedding_service import load_embeddings
from services.clustering_service import (
    find_optimal_k, cluster_documents, get_cluster_summary,
    save_clustering, assign_query_to_cluster
)

print('Loading CT2021 embeddings...')
doc_ids1, emb1 = load_embeddings(f'{SAVE_DIR}/ct2021')
print(f'Loaded: {emb1.shape}')

In [ ]:
# Find optimal number of clusters
print('Finding optimal K (testing K=5 to 15)...')
opt = find_optimal_k(emb1, k_range=range(5, 16), sample_size=20000)
print(f'Best K by silhouette score: {opt["best_k"]}')
print('Silhouette scores:', {k: round(v, 4) for k, v in opt['silhouette_scores'].items()})

In [ ]:
# Cluster CT2021 documents
N_CLUSTERS = opt['best_k']
print(f'Clustering {len(doc_ids1):,} documents into {N_CLUSTERS} clusters...')
labels1, km1 = cluster_documents(emb1, n_clusters=N_CLUSTERS)

summary1 = get_cluster_summary(doc_ids1, labels1, index1, N_CLUSTERS)
print('\n=== CT2021 Cluster Summary ===')
for cid, info in summary1.items():
    print(f'Cluster {cid}: {info["size"]:,} docs | Top terms: {info["top_terms"][:5]}')

save_clustering(labels1, km1, f'{SAVE_DIR}/ct2021')
print('\nClustering saved')

In [ ]:
# Cluster MSMARCO
from services.clustering_service import load_clustering
doc_ids2, emb2 = load_embeddings(f'{SAVE_DIR}/msmarco')
index2 = load_index(f'{SAVE_DIR}/msmarco_index.pkl')

print(f'Clustering MSMARCO {len(doc_ids2):,} docs into {N_CLUSTERS} clusters...')
labels2, km2 = cluster_documents(emb2, n_clusters=N_CLUSTERS)
summary2 = get_cluster_summary(doc_ids2, labels2, index2, N_CLUSTERS)

for cid, info in summary2.items():
    print(f'Cluster {cid}: {info["size"]:,} docs | Top terms: {info["top_terms"][:5]}')

save_clustering(labels2, km2, f'{SAVE_DIR}/msmarco')
with open(f'{SAVE_DIR}/ct2021_cluster_summary.json', 'w') as f:
    json.dump(summary1, f, indent=2)
with open(f'{SAVE_DIR}/msmarco_cluster_summary.json', 'w') as f:
    json.dump(summary2, f, indent=2)
print('Clustering complete')

## Part 3 — RAG (Retrieval-Augmented Generation)

In [ ]:
from services.bm25_service import retrieve_bm25, load_bm25_params
from services.rag_service import load_rag_model, generate_answer

print('Loading RAG model (flan-t5-base)...')
rag_pipeline = load_rag_model()

with open(f'{SAVE_DIR}/ct2021_docs_processed.json') as f:
    raw_docs1 = json.load(f)

bm25_params1 = load_bm25_params(f'{SAVE_DIR}/ct2021_bm25_params.pkl')
avg_dl1 = bm25_params1['avg_dl']
with open(f'{SAVE_DIR}/ct2021_doc_lengths.json') as f:
    doc_lengths1 = json.load(f)
print('RAG ready')

In [ ]:
# Test RAG on sample queries
test_qs = [
    'What are the effects of metformin on diabetes patients?',
    'Clinical trials for lung cancer immunotherapy',
]

for q in test_qs:
    tokens = preprocess(q)
    retrieved = retrieve_bm25(tokens, index1, doc_lengths1, avg_dl1, top_k=5)
    result = generate_answer(q, retrieved, raw_docs1, pipeline=rag_pipeline, max_docs=5)
    print(f'Query: {q}')
    print(f'Answer: {result["answer"]}')
    print(f'Sources: {result["sources"]}')
    print()

## Part 4 — Learning to Rank (LTR)

In [ ]:
from services.ltr_service import (
    build_feature_matrix_multi, train_ltr_model,
    rerank_with_ltr, evaluate_ltr, save_ltr_model, load_ltr_model
)
from services.evaluation_service import load_qrels, evaluate_run, print_results_table

# Load all retrieval results for CT2021
def load_res(path):
    with open(path) as f: return json.load(f)

results_per_model1 = {
    'bm25':      load_res(f'{SAVE_DIR}/ct2021_bm25_results.json'),
    'tfidf':     load_res(f'{SAVE_DIR}/ct2021_tfidf_results.json'),
    'embedding': load_res(f'{SAVE_DIR}/ct2021_embedding_results.json'),
}

ds1 = ir_datasets.load('clinicaltrials/2021/trec-ct-2021')
qrels1 = load_qrels(ds1)
query_ids1 = list(qrels1.keys())
print(f'Training LTR on CT2021: {len(query_ids1)} queries')

In [ ]:
# Split train/test (70/30)
split = int(len(query_ids1) * 0.7)
train_qids = query_ids1[:split]
test_qids  = query_ids1[split:]

model_names = list(results_per_model1.keys())

X_train, y_train, _ = build_feature_matrix_multi(train_qids, results_per_model1, qrels1)
X_test,  y_test,  _ = build_feature_matrix_multi(test_qids,  results_per_model1, qrels1)
print(f'Train: {X_train.shape}, positives: {y_train.sum()}')
print(f'Test:  {X_test.shape},  positives: {y_test.sum()}')

ltr1 = train_ltr_model(X_train, y_train)
train_metrics = evaluate_ltr(X_train, y_train, ltr1)
test_metrics  = evaluate_ltr(X_test,  y_test,  ltr1)
print(f'LTR Train — Accuracy: {train_metrics["accuracy"]}, AUC: {train_metrics["auc"]}')
print(f'LTR Test  — Accuracy: {test_metrics["accuracy"]},  AUC: {test_metrics["auc"]}')
save_ltr_model(ltr1, f'{SAVE_DIR}/ct2021_ltr_model.pkl')

In [ ]:
# Compare LTR vs baseline on test queries
queries1 = {q.query_id: q.text for q in ds1.queries_iter()}

ltr_results1 = {}
for qid in test_qids:
    ltr_results1[qid] = rerank_with_ltr(qid, results_per_model1, ltr1, model_names, top_k=1000)

ltr_eval   = evaluate_run(ltr_results1, qrels1)['aggregated']
bm25_eval  = evaluate_run({q: results_per_model1['bm25'][q]  for q in test_qids if q in results_per_model1['bm25']},  qrels1)['aggregated']
emb_eval   = evaluate_run({q: results_per_model1['embedding'][q] for q in test_qids if q in results_per_model1['embedding']}, qrels1)['aggregated']

print('\n=== LTR vs Baselines (CT2021 test set) ===')
print_results_table({'BM25': bm25_eval, 'Embedding': emb_eval, 'LTR': ltr_eval})

with open(f'{SAVE_DIR}/ct2021_ltr_results.json', 'w') as f:
    json.dump(ltr_results1, f)
print('\nLTR results saved')
print('\n=== Additional Features Complete ===')